In [1]:
import xtrack as xt
import sys
xutil_path = f'../helper_functions' # specify the path to the helper_functions folder
sys.path.insert(0, xutil_path)
from helpers_from_xutil import set_beam_beam_scale, install_beam_beam_elements, set_BBelem_shift

In [2]:
# CHOOSE SEED HERE
seed = 4
# CHOOSE LINE VERSION HERE
line_version = "LCC_105-0-0_z"
# ADJUST THE FOLDER PATH BELOW TO MATCH YOUR DIRECTORY STRUCTURE
folder_path = f'../lattices/V105'

In [3]:
# Load line with corrected imperfections
line = xt.Line.from_json(f'{folder_path}/lattices_with_corrected_imperfections/03_orbit_and_optics_corrected_with_radiation_FINAL/{line_version}_line_corrected_orbit_and_optics_seed{seed}.json')
# The corrected lines have 6d Twiss method, radiation and tapering enabled, and are cycled to rf cavity

# Good to have this just to be safe
line.cycle(name_first_element='rf400', inplace=True)
line.twiss_default['method'] = '6d'
line.configure_radiation(model='mean', model_beamstrahlung=None)
line.compensate_radiation_energy_loss()

Loading line from dict: 100%|██████████| 21288/21288 [00:01<00:00, 13559.60it/s]


Done loading line from dict.           
Compensating energy loss.
Share energy loss among cavities (repeat until energy loss is zero)
Energy loss: 34_700_409.678 eV             
Energy loss: 105_623.666 eV             
Energy loss: 321.872 eV             
Energy loss: 0.981 eV             
Energy loss: -53_647.638 eV             
Energy loss: -244.957 eV             
Energy loss: 162.181 eV             
Energy loss: 1.483 eV             

  - Set delta_taper
  - Restore cavity voltage and frequency. Set cavity lag


In [4]:
tt = line.get_table()
ttips = tt.rows['ip.*']
ttips = ttips.rows[[0, 2, 4, 6]] # store the table rows for the 4 IPs only

In [5]:
# Define parameters for beam-beam interaction
reference_parameters = {
    'normalized_emittance_x': 0.7e-9 * 89236, # relativistic gamma factor is 89236
    'normalized_emittance_y': 1.4e-12 * 89236, 
    'bunch_length': 16.7e-3,
    'bunch_population': 2.02e11}
beam_beam_parameters = {
    'collisions' : True,
    'num_IPs' : 4,
    'half_xing_angle' : 15*1e-3, # half-crossing angle in radians
    'xing_plane' : 0,
    'num_slices' : 251,
    'beamstrahlung_on' : True,
    'binning_mode' : 'unicharge'}

In [ ]:
# Install beam-beam elements at zero strength
bb_elem_names = install_beam_beam_elements(line, reference_parameters, beam_beam_parameters, ip_list=list(ttips.name))
set_beam_beam_scale(line, 0.0, ip_list = list(ttips.name))

# Ensure the BB elems are correctly shifted at the IPs (very important!)
tw = line.twiss(coupling_edw_teng=True, matrix_stability_tol=0.20)
set_BBelem_shift(line, tw) 

print(f"Installed {len(bb_elem_names)} beam-beam elements and shifted them correctly!")

/home/larasievert/miniforge3/envs/fcc-2026/lib/python3.13/site-packages/xtrack/line.py:3330: FutureWarning: Line.insert_element is deprecated. Use Line.insert instead. This deprecation is part of the interface cleanup in view of the 1.0 release.
  warn('Line.insert_element is deprecated. Use Line.insert instead.'


Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Installed 4 beam-beam elements and shifted them correctly !


In [7]:
# Set beam-beam to the chosen scale strength
BB_scale_strength = 0.95
set_beam_beam_scale(line, BB_scale_strength, ip_list=list(ttips.name))